In [ ]:
import json
from transformers.utils import get_json_schema
# =====================================================================
# PASO 1: DEFINICIÓN DE LA FUNCIÓN REAL EN PYTHON
# =====================================================================
def multiply(a: float, b: float) -> float:
    """
    A function that multiplies two numbers

    Args:
        a: The first number to multiply
        b: The second number to multiply
    """
    return a * 25

# Mapeo físico para asociar el string que devuelve el LLM con la función real
available_tools = {
    "multiply": multiply
}
tools = get_json_schema(multiply)
# Esquema JSON estructurado siguiendo el estándar estricto de OpenAI Tools
'''
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "A function that multiplies two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "The first number to multiply."},
                    "b": {"type": "number", "description": "The second number to multiply."}
                },
                "required": ["a", "b"]
            },
            "return": {
                "type": "number"
            }
        }
    }
]
'''
# print(tools)
# =====================================================================
# PASO 2: INICIALIZACIÓN DEL HISTORIAL DE LA CONVERSACIÓN
# =====================================================================
first_message = messages = [
    {"role": "system", "content": "You are a helpful assistant with access to mathematical tools."},
    {"role": "user", "content": "Hey, can you multiply 4.5 by 3 for me?"}
]

Si invocamos el modelo localmente la respuesta es un XML que usa una etiqueta \<tool_call\>. Habría que parsearla y es un rollo así que usamos openai para que lo parsee a un objeto.

In [ ]:
from llama_cpp import Llama
llm = Llama(
    model_path="/home/roberto/Applications/models/Qwen/Qwen3-1.7B-GGUF/Qwen3-1.7B-Q8_0.gguf",
    vocab_only=False, # Si solo queremos formatear el prompt lo ponemos a False
    verbose=False
)
# =====================================================================
# PASO 3: PRIMERA LLAMADA (Generación estructurada controlada por Gramática)
# =====================================================================
# llama_cpp_python emula el formato de la API de OpenAI de forma nativa
response = llm.create_chat_completion(
    messages=first_message,
    tools=tools,
    tool_choice="auto",
    temperature=0.1     # Temperatura baja para maximizar la precisión
)
response_message = response["choices"][0]["message"]
print(response_message)

In [ ]:

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1", 
    api_key="token-o-api-key-de-tu-servidor" # Si tu servidor local no pide key, pon cualquier texto
)

# =====================================================================
# PASO 3: PRIMERA LLAMADA (El servidor procesa y decide usar la Tool)
# =====================================================================
response = client.chat.completions.create(
    model="qwen3",
    messages=first_message,
    tools=[tools],
    tool_choice="auto", # El modelo decide libremente si usar la función o responder texto
    temperature=0.1     # Mantener baja para asegurar la precisión del JSON
)

response_message = response.choices[0].message
print(response_message)

Observa que es necesario añadir los mensajes anteriores. Cada vez que invocamos el modelo este olvida la conversación anterior.

In [ ]:
# =====================================================================
# PASO 4: PARSEO Y EJECUCIÓN DINÁMICA DE LA HERRAMIENTA
# =====================================================================
# Verificamos si el modelo decidió invocar alguna herramienta
if response_message.tool_calls:
    print("\n🤖 [El modelo solicitó ejecutar una función]")
    # Añadimos la intención del asistente a la lista histórica de mensajes
    messages.append(response_message)
    for tool_call in response_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        print(f"👉 Nombre de la función: {function_name}")
        print(f"👉 Argumentos proporcionados: {function_args}")
        
        # Ejecutamos la función localmente en nuestro entorno de Python
        if function_name in available_tools:
            function_to_call = available_tools[function_name]
            # Ejecución real: multiply(a=4.5, b=3)
            tool_output = function_to_call(**function_args) 
            
            print(f"✅ [Resultado de la ejecución local]: {tool_output}")
            
            # =====================================================================
            # PASO 5: ENVIAR EL RESULTADO DE REGRESO AL SERVIDOR
            # =====================================================================
            # Es obligatorio incluir el 'tool_call_id' para que el servidor
            # sepa exactamente a qué llamada corresponde esta respuesta.
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(tool_output)
            })
        else:
            print(f"❌ Error: La función '{function_name}' no existe en las herramientas registradas.")
            
    print("\n🔄 Enviando el resultado final de la herramienta de vuelta a Qwen...")
    
    # Realizamos la segunda llamada enviando todo el historial con el resultado de la tool
    final_response = client.chat.completions.create(
        model="qwen3",
        messages=messages
    )
    
    final_text = final_response.choices[0].message.content
    print(f"\n🏁 [Respuesta Final de Qwen]:\n{final_text}")

else:
    # Si el usuario no pidió matemáticas y el modelo respondió texto plano directo
    print(f"\n🤖 [Respuesta Directa de Qwen]:\n{response_message.content}")


In [1]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Definimos la estructura y tipo de los datos (Pydantic)
class Usuario(BaseModel):
    id: int
    nombre: str

# Endpoint básico (Asíncrono y nativo)
@app.get("/")
async def home():
    return {"mensaje": "¡Hola desde FastAPI!"}

# Endpoint con datos (Validación Automática)
@app.post("/usuarios", status_code=201)
async def crear_usuario(usuario: Usuario):
    # FastAPI ya validó que 'id' sea int y 'nombre' sea str. 
    # Si el cliente envía datos mal estructurados, recibe un Error 422 automáticamente.
    return {
        "status": "Usuario creado",
        "usuario": usuario
    }

In [2]:
datos_nuevo_usuario = {
    "id": 42,
    "nombre": "Carlos Alberto"
}

# Enviamos un ID inválido a propósito ("no_soy_un_numero")
datos_erroneos = {
    "id": "no_soy_un_numero",
    "nombre": "Marta"
}

In [3]:
from fastapi.testclient import TestClient
# El TestClient permite hacer peticiones simuladas a 'app' sin levantar Uvicorn
cliente = TestClient(app)

# 1. Enviamos la petición GET
respuesta = cliente.get("/")
print(respuesta.json())

# 2. Enviamos la petición POST incluyendo el JSON
respuesta = cliente.post("/usuarios", json=datos_nuevo_usuario)

# 3. Imprimimos el código de estado (debería ser 201) y la respuesta del servidor
print(f"Código de estado: {respuesta.status_code}")
print("Respuesta del servidor:")
print(respuesta.json())

{'mensaje': '¡Hola desde FastAPI!'}
Código de estado: 201
Respuesta del servidor:
{'status': 'Usuario creado', 'usuario': {'id': 42, 'nombre': 'Carlos Alberto'}}


In [19]:
# Celda 2: Lanzamos Gradio (Él se encarga del hilo en segundo plano)
from fastapi.testclient import TestClient
import gradio as gr

# El TestClient permite hacer peticiones simuladas a 'app' sin levantar Uvicorn
cliente = TestClient(app)

# 1. Función para probar el GET
def probar_get():
    respuesta = cliente.get("/")
    return respuesta.json()

# 2. Función para probar el POST
def probar_post(id_user, nombre_user):
    payload = {"id": int(id_user), "nombre": nombre_user}
    respuesta = cliente.post("/usuarios", json=payload)
    return f"Código {respuesta.status_code}\n{respuesta.json()}"

# 3. Diseñamos la interfaz visual con dos pestañas de prueba
with gr.Blocks(title="Probador de FastAPI") as demo:
    gr.Markdown("### 🚀 Probador Visual para tu Servidor FastAPI")
    
    with gr.Tab("Probar GET (/)"):
        boton_get = gr.Button("Enviar GET")
        salida_get = gr.JSON()
        boton_get.click(fn=probar_get, inputs=[], outputs=salida_get)
        
    with gr.Tab("Probar POST (/usuarios)"):
        input_id = gr.Number(label="ID de Usuario")
        input_nombre = gr.Textbox(label="Nombre de Usuario")
        boton_post = gr.Button("Enviar POST")
        salida_post = gr.Textbox(label="Resultado")
        boton_post.click(fn=probar_post, inputs=[input_id, input_nombre], outputs=salida_post)

# Lanzamiento automático en segundo plano gracias a Gradio
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [20]:
demo.close()

Closing server running on port: 7860


In [21]:
ip_usuarios = "127.0.0.1"
#ip_usuarios = "0.0.0.0"
# Para ejecutar: uvicorn nombre_archivo:app --host 127.0.0.1 --port 5000 --reload
# o bien incluir en el script la linea
'''
if __name__ == "__main__":
    uvicorn.run("nombre_archivo:app", host="0.0.0.0", port=5000, reload=True)
'''
# y lanzar el script con python nombre_archivo.py
# 2. Corremos uvicorn directamente indicando una configuración básica
# Nota: En Jupyter suele ser mejor usar una función asíncrona o configurar un puerto libre.
import uvicorn
# ==========================================
# 1. PARCHEAR JUPYTER (Obligatorio)
# ==========================================
# Jupyter ya tiene un servidor corriendo. Este parche permite que Uvicorn 
# levante un segundo servidor web dentro del mismo bucle sin lanzar errores.
config = uvicorn.Config(app, host=ip_usuarios, port=5000, log_level="info")
server = uvicorn.Server(config)
# 3. Lanzamos el servidor en el hilo nativo
import threading
import time
server.install_signal_handlers = lambda *args, **kwargs: None
hilo_servidor = threading.Thread(target=server.run, daemon=True)
hilo_servidor.start()

INFO:     Started server process [12355]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)


In [28]:
import requests
url_usuarios = "http://" + ip_usuarios + ':5000'
try:
    # 1. Enviamos la petición GET
    respuesta = requests.get(url_usuarios, timeout=5)
    print(respuesta.json())
    
    # 2. Enviamos la petición POST incluyendo el JSON
    respuesta = requests.post(url_usuarios+'/usuarios', json=datos_nuevo_usuario, timeout=5)
    
    # 3. Imprimimos el código de estado (debería ser 201) y la respuesta del servidor
    print(f"Código de estado: {respuesta.status_code}")
    print("Respuesta del servidor:")
    print(respuesta.json())
except requests.exceptions.Timeout:
    print("❌ ¡Error! El servidor tardó demasiado en responder (Timeout).")
except requests.exceptions.ConnectionError:
    print("❌ ¡Error de conexión! No hay ningún servidor escuchando en esa IP/Puerto.")

❌ ¡Error de conexión! No hay ningún servidor escuchando en esa IP/Puerto.


In [25]:
# Gradio asume automáticamente que cuando el usuario pulsa el botón "Submit", se toman los datos de la izquierda, 
# se pasan a la función fn y el resultado se pinta a la derecha.
def probar_api_desde_gradio(id_usuario, nombre_usuario):
    """
    Esta función se ejecuta cuando pulsas el botón en Gradio.
    Toma los datos de la interfaz gráfica y hace un POST real a la API.
    """
    payload = {"id": int(id_usuario), "nombre": nombre_usuario}
    
    try:
        # Hacemos la llamada POST real a nuestra propia API de FastAPI
        respuesta = requests.post(url_usuarios+'/usuarios', json=payload)
        
        # Devolvemos el resultado formateado para mostrarlo en la pantalla de Gradio
        return f"Código HTTP: {respuesta.status_code}\nRespuesta JSON: {respuesta.json()}"
    except Exception as e:
        return f"Error al conectar con la API: {str(e)}"

io = gr.Interface(
    fn=probar_api_desde_gradio, 
    inputs=["number", "text"], # Pide un ID (número) y un Nombre (texto)
    outputs="text",            # Muestra el resultado del POST en texto
    title="Probador Visual de Endpoints POST"
)

io.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [26]:
# Celda nueva para apagar el servidor de Gradio
io.close()
print("¡Servidor de Gradio e interfaz web detenidos!")

Closing server running on port: 7860
¡Servidor de Gradio e interfaz web detenidos!


In [27]:
server.should_exit = True
hilo_servidor.join()
print("Servidor FastAPI detenido exitosamente.")

Servidor FastAPI detenido exitosamente.


## MCP

In [1]:
from openai import OpenAI
import os

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

resp = client.responses.create(
    model="gpt-5",
    tools=[
        {
            "type": "mcp",
            "server_label": "temperature_mcp",
            "server_description": "MCP para dar la temperatura actual en una ciudad",
            "server_url": "https://cc10cce859b7.ngrok-free.app/mcp",
            "require_approval": "never",
        },
    ],
    input="¿Qué temperatura hace en Sevilla ahora?",
)

print(resp.output_text)

APIStatusError: Error code: 424 - {'error': {'message': "Error retrieving tool list from MCP server: 'temperature_mcp'. Http status code: 424 (Failed Dependency)", 'type': 'external_connector_error', 'param': 'tools', 'code': 'http_error'}}